In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
# import torch
# import torch.nn.functional as F
# from torch_geometric.nn import GraphConv
# from torch.nn import Linear
# from torch_geometric.nn import global_mean_pool
import urllib.request
from node2vec import Node2Vec

Create ppi human graph

In [8]:
df = pd.read_csv('bio-pathways-network.csv')

edges = list(zip(df['Gene ID 1'], df['Gene ID 2']))

G_human = nx.Graph()
G_human.add_edges_from(edges)

print("Human nodes:", len(G_human.nodes()), "edges:", len(G_human.edges()))

Human nodes: 21557 edges: 342353


Retrieve and convert yeast file to graph

In [9]:
urllib.request.urlretrieve(
    'http://snap.stanford.edu/deepnetbio-ismb/ipynb/yeast.edgelist',
    'yeast.edgelist'
)
yeast_file = "yeast.edgelist"
G_yeast = nx.read_edgelist(yeast_file)

print("Yeast nodes:", len(G_yeast.nodes()), "edges:", len(G_yeast.edges()))

Yeast nodes: 6526 edges: 532180


Graph Embedding, Labelling and feature extraction

In [10]:
def create_labels(G, threshold=5):
    labels = {}
    for node, deg in dict(G.degree()).items():
        labels[node] = 1 if deg >= threshold else 0
    return labels

labels_yeast = create_labels(G_yeast, threshold=5)
labels_human = create_labels(G_human, threshold=10)

#print graph label
print("Yeast labels:", list(labels_yeast.items())[:10])
print("Human labels:", list(labels_human.items())[:10])

Yeast labels: [('YLR418C', 1), ('YOL145C', 1), ('YOR123C', 1), ('YBR279W', 1), ('YML069W', 1), ('YGL244W', 1), ('YGL207W', 1), ('YER164W', 1), ('YIL035C', 1), ('YOR061W', 1)]
Human labels: [(1394, 1), (2778, 1), (6331, 1), (17999, 1), (122704, 1), (54460, 1), (2597, 1), (2911, 1), (4790, 1), (79155, 1)]


In [ ]:
# Optimized version for faster execution
def generate_embeddings_fast(G, dimensions=64):
    """
    Faster version of Node2Vec embeddings with reduced parameters
    """
    node2vec = Node2Vec(G, 
                        dimensions=dimensions, 
                        walk_length=10,      # Reduced from 30 to 10
                        num_walks=50,        # Reduced from 200 to 50  
                        workers=8,           # Increased workers
                        quiet=True)
    
    model = node2vec.fit(window=5,           # Reduced window size
                        min_count=1, 
                        batch_words=4)

    # Create DataFrame of embeddings
    emb_df = pd.DataFrame([model.wv.get_vector(str(node)) for node in G.nodes()])
    emb_df["node"] = list(G.nodes())
    return emb_df, model

# Test with the faster version
print("Running optimized embeddings...")
emb_yeast_fast, model_yeast_fast = generate_embeddings_fast(G_yeast)
print("Yeast embeddings completed!")

emb_human_fast, model_human_fast = generate_embeddings_fast(G_human)  
print("Human embeddings completed!")

print("Fast Embeddings (Yeast):", emb_yeast_fast.shape)
print("Fast Embeddings (Human):", emb_human_fast.shape)


Running optimized embeddings...


c:\Users\tiffa\anaconda3\envs\ppi\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
